# Code to generate the thesis plots

To generate different plots depending on different simulations change the simulation path before the .csv filename. The last two simulations do not have the final stats inside the .csv files because of the error reported by Francesco in the group chat, however the file packet_result.csv was correctly populated and it is the one used to generate the first two plots, so we can use those simulations anyway. If you try you see that the results are very close to ones presented in the thesis.

In [1]:
shortest_low = "project/simulations/shortest_7.5/"
shortest_heavy = "project/simulations/shortest_1.5/"

between_low = "project/simulations/betweenness_7.5/"
between_heavy = "project/simulations/betweenness_1.5/"

greedy_low = "project/simulations/greedy_7.5/"
greedy_heavy = "project/simulations/greedy_1.5/"

graph_path = "project/file/graph_networkx_15_nodes.json"

# Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
import json

## Heatmap of the fraction of bits received by each node from the other nodes in the network

In [4]:
# It is the same as received bits, because we would multiply by 128 on both numerator and denumerator so it doesn't change anything.
def extract_fraction_received_packets(file):
    df = pd.read_csv(file)
    df["Sent"] = df["Sent"].astype(bool)
    df["Delivered"] = df["Delivered"].astype(bool)

    df_sent = df[df["Sent"] == True]

    # Group by Source and Destination and compute sent and delivered counts
    grouped = df_sent.groupby(["Source", "Destination"]).agg(
        sent_packets=("Sent", "count"),
        delivered_packets=("Delivered", "sum")
    ).reset_index()

    # Compute fraction of delivered over sent
    grouped["fraction_received"] = grouped["delivered_packets"] / grouped["sent_packets"]

    return grouped   # <-- return the full DataFrame, not just the Series


def heatmap(grouped, title: str, filename: str):
    heatmap_data = grouped.pivot(index="Destination", columns="Source", values="fraction_received")
    heatmap_data = heatmap_data.fillna(0)

    # Strip 'node' prefix from rows and columns
    heatmap_data.index = heatmap_data.index.str.replace("node", "", regex=False).astype(int)
    heatmap_data.columns = heatmap_data.columns.str.replace("node", "", regex=False).astype(int)

    # Sort rows and columns
    heatmap_data = heatmap_data.sort_index(axis=0)  # rows
    heatmap_data = heatmap_data.sort_index(axis=1)  # columns

    heatmap_data = heatmap_data.reindex(index=heatmap_data.index[::-1])

    cmap = mcolors.LinearSegmentedColormap.from_list("white_to_darkblue", ["white", "#00008B"])
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Fraction of bits received"}
    )

    plt.title(f"Fraction of bits received (Destination vs Source), {title}")
    plt.xlabel("Source Node")
    plt.ylabel("Destination Node")
    plt.tight_layout()
    plt.savefig(f"project/plots/heatmap_{filename}.png")
    plt.close()


def compare_heatmaps(grouped1, title1: str, grouped2, title2: str):
    """Display two heatmaps side by side for comparison."""
    def prepare_heatmap_data(grouped):
        heatmap_data = grouped.pivot(index="Destination", columns="Source", values="fraction_received")
        heatmap_data = heatmap_data.fillna(0)
        heatmap_data.index = heatmap_data.index.str.replace("node", "", regex=False).astype(int)
        heatmap_data.columns = heatmap_data.columns.str.replace("node", "", regex=False).astype(int)
        heatmap_data = heatmap_data.sort_index(axis=0)
        heatmap_data = heatmap_data.sort_index(axis=1)
        heatmap_data = heatmap_data.reindex(index=heatmap_data.index[::-1])
        return heatmap_data

    data1 = prepare_heatmap_data(grouped1)
    data2 = prepare_heatmap_data(grouped2)

    cmap = mcolors.LinearSegmentedColormap.from_list("white_to_darkblue", ["white", "#00008B"])

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    sns.heatmap(
        data1,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Fraction of bits received"},
        ax=axes[0]
    )
    axes[0].set_title(title1)
    axes[0].set_xlabel("Source Node")
    axes[0].set_ylabel("Destination Node")

    sns.heatmap(
        data2,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Fraction of bits received"},
        ax=axes[1]
    )
    axes[1].set_title(title2)
    axes[1].set_xlabel("Source Node")
    axes[1].set_ylabel("Destination Node")

    plt.tight_layout()
    plt.show()


bits_shortest_low = extract_fraction_received_packets(shortest_low + "packet_result.csv")
bits_between_low = extract_fraction_received_packets(between_low + "packet_result.csv")
bits_greedy_low = extract_fraction_received_packets(greedy_low + "packet_result.csv")
bits_shortest_heavy = extract_fraction_received_packets(shortest_heavy + "packet_result.csv")
bits_between_heavy = extract_fraction_received_packets(between_heavy + "packet_result.csv")
bits_greedy_heavy = extract_fraction_received_packets(greedy_heavy + "packet_result.csv")

heatmap(bits_shortest_low, "Shortest Path (Low Load)", "heatmap_shortest_low")
heatmap(bits_shortest_heavy, "Shortest Path (Heavy Load)", "heatmap_shortest_heavy")
heatmap(bits_between_low, "Edge Betweenness (Low Load)", "heatmap_between_low")
heatmap(bits_between_heavy, "Edge Betweenness (Heavy Load)", "heatmap_between_heavy")
heatmap(bits_greedy_low, "Greedy Algorithm (Low Load)", "heatmap_greedy_low")
heatmap(bits_greedy_heavy, "Greedy Algorithm (Heavy Load)", "heatmap_greedy_heavy")


'''
heatmap(short_low, "shortest low load")
heatmap(centflow_low, "centflow low load")
heatmap(short_heavy, "shortest heavy load")
heatmap(centflow_heavy, "centflow heavy load")
'''


print("Average of received packets for shortest with low load: " + str(bits_shortest_low["fraction_received"].mean()))
print("Average of received packets for between with low load: " + str(bits_between_low["fraction_received"].mean()))
print("Average of received packets for greedy with low load: " + str(bits_greedy_low["fraction_received"].mean()))
print("Average of received packets for shortest with heavy load: " + str(bits_shortest_heavy["fraction_received"].mean()))
print("Average of received packets for between with heavy load: " + str(bits_between_heavy["fraction_received"].mean()))
print("Average of received packets for greedy with heavy load: " + str(bits_greedy_heavy["fraction_received"].mean()))

Average of received packets for shortest with low load: 0.8998670785588334
Average of received packets for between with low load: 0.9408726413443825
Average of received packets for greedy with low load: 0.9347555080178745
Average of received packets for shortest with heavy load: 0.7218234445128712
Average of received packets for between with heavy load: 0.7229827328548911
Average of received packets for greedy with heavy load: 0.7029240535486289


## Number of received bits for each node

In [5]:
def extract_num_received_bits(file):
    df_packets = pd.read_csv(file)
    df_packets["Destination"] = df_packets["Destination"].str.extract(r"(\d+)", expand=False).astype(int)

    # Filter only delivered packets
    delivered_packets = df_packets[df_packets["Delivered"] == True]

    # Count received packets per destination node
    received_counts = delivered_packets["Destination"].value_counts().sort_index()
    received_bits = received_counts * 128

    # print("Number of correctly received packets per destination node:")
    # print(received_counts)
    return received_bits


def plot_num_received_bits(received_bits, title: str, filename: str):
    plt.figure(figsize=(10, 6))
    received_bits.plot(kind="bar", color="cornflowerblue")
    plt.title(f"Bits Correctly Received per Destination Node, {title}")
    plt.xlabel("Destination Node")
    plt.ylabel("Number of Bits")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"project/plots/received_bits_{filename}")
    plt.close()


def plot_comparison(recv_bits1, label1, recv_bits2, label2, recv_bits3, label3, recv_bits4, label4, title):
    # Ensure both Series have the same index order
    all_nodes = sorted(set(recv_bits1.index) | set(recv_bits2.index) | set(recv_bits3.index) | set(recv_bits4.index))
    vals1 = recv_bits1.reindex(all_nodes, fill_value=0)
    vals2 = recv_bits2.reindex(all_nodes, fill_value=0)
    vals3 = recv_bits3.reindex(all_nodes, fill_value=0)
    vals4 = recv_bits4.reindex(all_nodes, fill_value=0)

    x = np.arange(len(all_nodes))  # positions on x-axis
    width = 0.10                   # width of each bar

    plt.figure(figsize=(12, 6))
    bar_width = width / 4

    plt.bar(x - 3*bar_width/2, vals1, bar_width, label=label1, color="cornflowerblue")
    plt.bar(x - bar_width/2,   vals2, bar_width, label=label2, color="orange")
    plt.bar(x + bar_width/2,   vals3, bar_width, label=label3, color="red")
    plt.bar(x + 3*bar_width/2, vals4, bar_width, label=label4, color="green")

    plt.title(f"Bits Correctly Received per Destination Node")
    plt.xlabel("Destination Node")
    plt.ylabel("Number of Bits")
    plt.xticks(x, all_nodes)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()


shortest_low_rcv_bits = extract_num_received_bits(f"{shortest_low}packet_result.csv")
shortest_heavy_rcv_bits = extract_num_received_bits(f"{shortest_heavy}packet_result.csv")
between_low_rcv_bits = extract_num_received_bits(f'{between_low}packet_result.csv')
between_heavy_rcv_bits = extract_num_received_bits(f'{between_heavy}packet_result.csv')
greedy_low_rcv_bits = extract_num_received_bits(f'{greedy_low}packet_result.csv')
greedy_heavy_rcv_bits = extract_num_received_bits(f'{greedy_heavy}packet_result.csv')

plot_num_received_bits(shortest_low_rcv_bits, "Shortest Path Low Load", "shortest_low")
plot_num_received_bits(shortest_heavy_rcv_bits, "Shortest Path Heavy Load", "shortest_heavy")
plot_num_received_bits(between_low_rcv_bits, "Edge Betweenness Low Load", "between_low")
plot_num_received_bits(between_heavy_rcv_bits, "Edge Betweenness Heavy Load", "between_heavy")
plot_num_received_bits(greedy_low_rcv_bits, "Greedy Algorithm Low Load", "greedy_low")
plot_num_received_bits(greedy_heavy_rcv_bits, "Greedy Algorithm Heavy Load", "greedy_heavy")


# TO DO WHEN WE HAVE THE SIMULATIONS OF THE CENTFLOW
# Low load comparison
# plot_comparison(shortest_low_rcv_bits, "Shortest Path", 
#                 between_low_rcv_bits, "Edge Betweenness", 
#                 greedy_low_rcv_bits, "Greedy Algorithm",
#                 "Low Load")

# Heavy load comparison
# plot_comparison(shortest_heavy_rcv_bits, "Shortest Path", 
#                 between_heavy_rcv_bits, "Edge Betweenness", 
#                 greedy_heavy_rcv_bits, "Greedy Algorithm",
#                 "Heavy Load")

## Average of correctly received bits by destination nodes for paths of different lengths

Computing the unique path lengths in the experiments.

In [8]:
def extract_unique_path_lengths(csv_file):
    df = pd.read_csv(csv_file)
    
    # The column may contain spaces, so ensure we refer to it correctly:
    # Either 'Num. Hop' or something similar — check the header exactly.
    # If needed, strip leading/trailing whitespace:
    df.columns = df.columns.str.strip()
    
    if "Num. Hop" not in df.columns:
        raise ValueError("Column 'Num. Hop' not found in CSV file.")
    
    unique_hops = sorted([int(x) for x in df["Num. Hop"].unique()])
    return unique_hops


len_shortest_low = extract_unique_path_lengths(f"{shortest_low}packet_result.csv")
len_between_low = extract_unique_path_lengths(f'{between_low}packet_result.csv')
len_greedy_low = extract_unique_path_lengths(f'{greedy_low}packet_result.csv')
len_shortest_heavy = extract_unique_path_lengths(f"{shortest_heavy}packet_result.csv")
len_between_heavy = extract_unique_path_lengths(f'{between_heavy}packet_result.csv')
len_greedy_heavy = extract_unique_path_lengths(f'{greedy_heavy}packet_result.csv')

print("Path lenghts for shortest in low load: " + str(len_shortest_low))
print("Path lenghts for between in low load: " + str(len_between_low))
print("Path lenghts for greedy in low load: " + str(len_greedy_low))
print("Path lenghts for shortest in heavy load: " + str(len_shortest_heavy))
print("Path lenghts for between in heavy load: " + str(len_between_heavy))
print("Path lenghts for greedy in heavy load: " + str(len_greedy_heavy))

Path lenghts for shortest in low load: [0, 1, 2, 3, 4]
Path lenghts for between in low load: [0, 1, 2, 3, 4]
Path lenghts for greedy in low load: [0, 1, 2, 3, 4, 5]
Path lenghts for shortest in heavy load: [0, 1, 2, 3, 4]
Path lenghts for between in heavy load: [0, 1, 2, 3, 4]
Path lenghts for greedy in heavy load: [0, 1, 2, 3, 4, 5]


## Success probability for each path length

In [14]:
# Calculate and plot success probability for each path length

def extract_success_probability_by_path_length(packet_file, graph_path):
    # --- Load packet results ---
    df = pd.read_csv(packet_file)
    df["Sent"] = df["Sent"].astype(bool)
    df["Delivered"] = df["Delivered"].astype(bool)

    # --- Load graph and compute shortest path lengths ---
    with open(graph_path) as f:
        data = json.load(f)

    G = nx.node_link_graph(data, edges="links")  # type: ignore
    all_pairs_lengths = dict(nx.all_pairs_shortest_path_length(G))

    # --- Build mapping from (source, dest) to path length using string keys without 'node' ---
    path_length_dict = {}
    for src, targets in all_pairs_lengths.items():
        src_str = str(src).replace("node", "")
        for dst, length in targets.items():
            if src != dst:
                dst_str = str(dst).replace("node", "")
                path_length_dict[(src_str, dst_str)] = length

    # --- Add path length to each packet row ---
    def get_path_length(row):
        src = str(row["Source"]).replace("node", "")
        dst = str(row["Destination"]).replace("node", "")
        path_len = path_length_dict.get((src, dst))
        if path_len is None:
            # Optionally warn if a packet's source-dest pair is missing in the graph
            # print(f"Warning: path length missing for {src} -> {dst}")
            return None
        return path_len

    df["path_length"] = df.apply(get_path_length, axis=1)

    # Count how many packets have missing path length (debugging)
    missing = df["path_length"].isna().sum()
    if missing > 0:
        print(f"Warning: {missing} packets have missing path length")

    # --- Filter only sent packets ---
    df_sent = df[df["Sent"] == True]

    # --- Group by path length and calculate success probability ---
    grouped = df_sent.groupby("path_length").agg(
        sent_packets=("Sent", "count"),
        delivered_packets=("Delivered", "sum")
    ).reset_index()

    grouped["success_probability"] = grouped["delivered_packets"] / grouped["sent_packets"]

    # Optional: sort by path_length ascending
    grouped = grouped.sort_values("path_length").reset_index(drop=True)

    return grouped  # DataFrame with columns: path_length, sent_packets, delivered_packets, success_probability


from matplotlib.ticker import MaxNLocator

def plot_success_probability(grouped, title: str, filename: str):
    plt.figure(figsize=(8, 5))
    plt.bar(grouped["path_length"], grouped["success_probability"], color="skyblue")
    plt.title(f"Success Probability by Path Length ({title})")
    plt.xlabel("Path Length")
    plt.ylabel("Success Probability")
    plt.ylim(0, 1.05)
    plt.grid(axis="y")
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.tight_layout()
    plt.savefig(f"project/plots/success_prob_{filename}.png")
    plt.close()


def plot_success_probability_comparison_4(grouped1, label1,
                                           grouped2, label2,
                                           grouped3, label3,
                                           grouped4, label4,
                                           title: str,
                                           filename: str):

    plt.figure(figsize=(12, 6))

    # Plot each series with different markers and line styles
    plt.plot(grouped1["path_length"], grouped1["success_probability"], marker="o", linestyle="-",
             label=label1, color="cornflowerblue")
    plt.plot(grouped2["path_length"], grouped2["success_probability"], marker="s", linestyle="--",
             label=label2, color="orange")
    plt.plot(grouped3["path_length"], grouped3["success_probability"], marker="^", linestyle="-.",
             label=label3, color="red")
    plt.plot(grouped4["path_length"], grouped4["success_probability"], marker="d", linestyle=":",
             label=label4, color="green")

    plt.title(f"Success Probability vs Path Length — {title}")
    plt.xlabel("Path Length")
    plt.ylabel("Success Probability")
    plt.ylim(0, 1.05)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"project/plots/{filename}.png")
    plt.close()


# --- Extract data ---
short_low_success = extract_success_probability_by_path_length(f"{shortest_low}packet_result.csv", graph_path)
between_low_success = extract_success_probability_by_path_length(f"{between_low}packet_result.csv", graph_path)
greedy_low_success = extract_success_probability_by_path_length(f"{greedy_low}packet_result.csv", graph_path)
short_heavy_success = extract_success_probability_by_path_length(f"{shortest_heavy}packet_result.csv", graph_path)
between_heavy_success = extract_success_probability_by_path_length(f"{between_heavy}packet_result.csv", graph_path)
greedy_heavy_success = extract_success_probability_by_path_length(f"{greedy_heavy}packet_result.csv", graph_path)

# Plots
# plot_success_probability(short_low_success, "Shortest Path Low Load", "shortest_low")
# plot_success_probability(between_low_success, "Edge Betweenness Low Load", "between_low")
# plot_success_probability(greedy_low_success, "Greedy Algorithm Low Load", "greedy_low")
# plot_success_probability(short_heavy_success, "Shortest Path Heavy Load", "shortest_heavy")
# plot_success_probability(between_heavy_success, "Edge Betweenness Heavy Load", "between_heavy")
# plot_success_probability(greedy_heavy_success, "Greedy Algorithm Heavy Load", "greedy_heavy")

# --- Plot comparisons ---
# Low load comparison
# plot_success_probability_comparison(short_low_success, "Shortest Path",
#                                     between_low_success, "Edge Betweenness",
#                                     greedy_low_success, "Greedy Algorithm",
#                                     "Low Load")

# Heavy load comparison
# plot_success_probability_comparison(short_low_success, "Shortest Path",
#                                     between_heavy_success, "Edge Betweenness",
#                                     greedy_heavy_success, "Greedy Algorithm",
#                                     "Heavy Load")

In [16]:
print(greedy_heavy_success)
print(greedy_low_success)

   path_length  sent_packets  delivered_packets  success_probability
0            1         12270              12154             0.990546
1            2         19403              12334             0.635675
2            3          9421               5037             0.534657
3            4          1157                685             0.592048
   path_length  sent_packets  delivered_packets  success_probability
0            1         29451              29451             1.000000
1            2         59300              55109             0.929325
2            3         38584              34543             0.895267
3            4          5800               5249             0.905000
